In [1]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from panel_utils import ModelResultsAggregator, run_panel_regressions, run_spec_tests, run_panel_model_diagnostics


In [2]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')

# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])
 
# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)

df_reg['Cluster_3'] = ((df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)).astype(int)


# Взаимодействия общего шока
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock'] * df_reg['Sank_dum']


# Взаимодействия негативного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_neg'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_neg'] * df_reg['Sank_dum']

# Взаимодействия позитивного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_pos'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_pos'] * df_reg['Sank_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[df_reg['Cluster_3'] == 1].copy()





### Мелкая модель

In [3]:
# Лаги 
df_reg["d_Mon_Shock_lag1"] = df_reg["d_Mon_Shock"].shift(1)
df_reg["d_Ex_Rate_lag1"] = df_reg["d_Ex_Rate"].shift(1)

In [4]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

exog_vars_initial = [
    'Int_Rate_FL_lag1',
    # 'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    # 'Credit_impulse',
    # 'Cred_structure',
    'Def_Zadolg_ConsCred',
    # 'Mon_Shock',
    'd_Mon_Shock_pos',
    'd_Mon_Shock_neg',
    # 'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
]

dependent_var = 'd_Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0668
Estimator:                   PooledOLS   R-squared (Between):              0.4507
No. Observations:                 6237   R-squared (Within):               0.0643
Date:                 Thu, Jan 15 2026   R-squared (Overall):              0.0668
Time:                         09:31:54   Log-likelihood                -1.299e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      37.153
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(12,6225)
Min Obs:                        77.000                          

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1134: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                           PanelOLS Estimation Summary                           
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0675
Estimator:                    PanelOLS   R-squared (Between):             -6.2756
No. Observations:                 6237   R-squared (Within):               0.0675
Date:                 Thu, Jan 15 2026   R-squared (Overall):              0.0251
Time:                         09:31:54   Log-likelihood                -1.297e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      44.486
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(10,6146)
Min Obs:                        77.000                                           
Max Obs:                        77.000   F-statistic (robust):             396.85
                

In [5]:
# run_panel_model_diagnostics(
#     y, X, pooled_res, fe_res, re_res,
#     pooled_success, fe_success, re_success
# )


In [6]:
# # ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

# run_spec_tests(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )


In [7]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

exog_vars_initial = [
    'Int_Rate_FL_lag1',
    # 'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    # 'Credit_impulse',
    # 'Cred_structure',
    'Def_Zadolg_ConsCred',
    'ROISFIX',
    # 'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
]

dependent_var = 'd_Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0850
Estimator:                   PooledOLS   R-squared (Between):              0.3647
No. Observations:                 6237   R-squared (Within):               0.0831
Date:                 Thu, Jan 15 2026   R-squared (Overall):              0.0850
Time:                         09:31:54   Log-likelihood                -1.293e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      52.556
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(11,6226)
Min Obs:                        77.000                

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1134: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                           PanelOLS Estimation Summary                           
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0924
Estimator:                    PanelOLS   R-squared (Between):             -2271.1
No. Observations:                 6237   R-squared (Within):               0.0924
Date:                 Thu, Jan 15 2026   R-squared (Overall):             -15.080
Time:                         09:31:54   Log-likelihood                -1.288e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      69.493
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                  F(9,6147)
Min Obs:                        77.000                                           
Max Obs:                        77.000   F-statistic (robust):             161.19
                

In [8]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

exog_vars_initial = [
    'Int_Rate_FL_lag1',
    # 'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    # 'Credit_impulse',
    # 'Cred_structure',
    'Def_Zadolg_ConsCred',
    'MIACR',
    # 'Bonds_Rate_Correct_5Y',
    'Exc_rate',
    'Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
]

dependent_var = 'd_Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0851
Estimator:                   PooledOLS   R-squared (Between):              0.3460
No. Observations:                 6237   R-squared (Within):               0.0833
Date:                 Thu, Jan 15 2026   R-squared (Overall):              0.0851
Time:                         09:31:55   Log-likelihood                -1.293e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      52.650
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(11,6226)
Min Obs:                        77.000                          

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1134: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                         RandomEffects Estimation Summary                        
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.0851
Estimator:               RandomEffects   R-squared (Between):              0.3460
No. Observations:                 6237   R-squared (Within):               0.0833
Date:                 Thu, Jan 15 2026   R-squared (Overall):              0.0851
Time:                         09:31:55   Log-likelihood                -1.293e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      52.650
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(11,6226)
Min Obs:                        77.000                                           
Max Obs:                        77.000   F-statistic (robust):             43.992
                

### Вывод результатов

In [15]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_ConsCred'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель(м) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель(м) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель(м) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель(б) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель(б) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель(б) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_test.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

,Модель(м) шок Тейлора (POOL),Модель(м) шок Тейлора (FE),Модель(м) шок Тейлора (RE),Модель(м) ROISFIX (POOL),Модель(м) ROISFIX (FE),Модель(м) ROISFIX (RE),Модель(м) MIACR (POOL),Модель(м) MIACR (FE),Модель(м) MIACR (RE),Модель(б) шок Тейлора (POOL),Модель(б) шок Тейлора (FE),Модель(б) шок Тейлора (RE),Модель(б) ROISFIX (POOL),Модель(б) ROISFIX (FE),Модель(б) ROISFIX (RE),Модель(б) MIACR (POOL),Модель(б) MIACR (FE),Модель(б) MIACR (RE)
Зависимая переменная,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred
Int_Rate_FL_lag1,-0.057*** (0.000),-0.048*** (0.000),-0.057*** (0.000),-0.220*** (0.000),-0.241*** (0.000),-0.220*** (0.000),-0.221*** (0.000),-0.241*** (0.000),-0.221*** (0.000),-0.143*** (0.000),-0.118*** (0.000),-0.143*** (0.000),-0.228*** (0.000),-0.226*** (0.000),-0.228*** (0.000),-0.232*** (0.000),-0.229*** (0.000),-0.232*** (0.000)
D_top5_rozn,-2.031*** (0.000),-3.161*** (0.003),-2.031*** (0.000),1.505** (0.043),-5.793*** (0.000),1.505** (0.043),1.280* (0.076),-6.407*** (0.000),1.280* (0.076),0.516 (0.266),0.542 (0.488),0.516 (0.266),2.025*** (0.003),-1.876** (0.043),2.025*** (0.003),1.977*** (0.004),-2.362** (0.013),1.977*** (0.004)
Fin_Dostup,-0.003 (0.412),0.056*** (0.001),-0.003 (0.412),0.010* (0.072),-0.008 (0.485),0.010* (0.072),0.009* (0.091),-0.008 (0.497),0.009* (0.091),0.005 (0.183),0.031*** (0.005),0.005 (0.183),0.008** (0.042),-0.011 (0.356),0.008** (0.042),0.008** (0.037),-0.011 (0.343),0.008** (0.037)
Def_Zadolg_ConsCred,-0.071*** (0.000),-0.171*** (0.000),-0.071*** (0.000),0.029 (0.146),-0.106*** (0.000),0.029 (0.146),0.022 (0.256),-0.125*** (0.000),0.022 (0.256),,,,,,,,,
d_Mon_Shock_pos,0.059*** (0.000),0.065*** (0.000),0.059*** (0.000),,,,,,,0.044*** (0.000),0.045*** (0.000),0.044*** (0.000),,,,,,
d_Mon_Shock_neg,-0.314*** (0.000),-0.324*** (0.000),-0.314*** (0.000),,,,,,,-0.241*** (0.003),-0.253*** (0.000),-0.241*** (0.003),,,,,,
Exc_rate,0.034*** (0.000),0.034*** (0.000),0.034*** (0.000),0.014*** (0.002),0.003* (0.052),0.014*** (0.002),0.017*** (0.000),0.006*** (0.000),0.017*** (0.000),0.034*** (0.000),0.034*** (0.000),0.034*** (0.000),0.021*** (0.000),0.012*** (0.000),0.021*** (0.000),0.023*** (0.000),0.015*** (0.000),0.023*** (0.000)
Inflation_Expectations,0.041*** (0.000),0.069*** (0.000),0.041*** (0.000),-0.093*** (0.000),-0.115*** (0.000),-0.093*** (0.000),-0.083*** (0.000),-0.100*** (0.000),-0.083*** (0.000),-0.024 (0.110),-0.020** (0.031),-0.024 (0.110),-0.101*** (0.000),-0.123*** (0.000),-0.101*** (0.000),-0.095*** (0.000),-0.112*** (0.000),-0.095*** (0.000)
Covid_dum,-0.364*** (0.000),-0.306*** (0.000),-0.364*** (0.000),-0.111** (0.024),0.056*** (0.004),-0.111** (0.024),-0.134*** (0.005),0.034* (0.094),-0.134*** (0.005),-0.358*** (0.000),-0.309*** (0.000),-0.358*** (0.000),-0.200*** (0.000),-0.046** (0.036),-0.200*** (0.000),-0.210*** (0.000),-0.055** (0.018),-0.210*** (0.000)
